# PDelta3 Frontier Layer Lab

This notebook tests four focused recurrent-attention replacements against the frozen Transformer attention layer:

1. **`conv4_pdelta_f96`** — current Conv4 + PDelta2 F96 control.
2. **`conv4_channel_decay_f96`** — PDelta with stronger content-dependent per-feature KDA/GDN2-style decay.
3. **`conv4_gdn2_f96`** — GDN2-style independent key-channel erase and value-channel write gates, with short Conv4 on Q/K/V.
4. **`conv4_gdn2_clvr_f96`** — the GDN2 candidate plus one-hop **Cross-Layer Value Routing**: the previous layer's value representation is aligned and injected into the current write target.

The lab uses progressive **256 -> 512 -> 1024** adaptation, selects the architecture using a long-context validation score (512/1024/2048), and keeps the test set untouched until final evaluation at **512/1024/2048/4096**.

A strict Transformer quality win is reported only when the paired 95% bootstrap CI of `candidate NLL - Transformer NLL` is entirely below zero. The implementation is an independent TinyCeNN adaptation to frozen SmolLM2 Q/K/V, not a reproduction of full frontier-model pretraining or fused kernels.


In [ ]:
import os, sys, subprocess, tempfile
from pathlib import Path

assert subprocess.run(["nvidia-smi"], check=False).returncode == 0, "Enable a GPU runtime in Colab."
REPO = Path(tempfile.mkdtemp(prefix="TinyCeNN-pdelta3-"))
subprocess.run(["git", "clone", "--depth", "1", "https://github.com/vtavakkoli/TinyCeNN-LM.git", str(REPO)], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers==4.57.6", "datasets", "huggingface_hub", "pandas", "matplotlib"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO), "--no-deps"], check=True)
print("Repository:", REPO)
subprocess.run(["git", "-C", str(REPO), "rev-parse", "HEAD"], check=True)


In [ ]:
from datetime import datetime, timezone

PROFILE = "balanced"          # quick | balanced | strong
LAYER = 18
CURRICULUM = "256,512,1024"
VALIDATION_CONTEXTS = "512,1024,2048"
TEST_CONTEXTS = "512,1024,2048,4096"
SEED = 2026

RESULT_ROOT = Path("/content/TinyCeNN-pdelta3-results")
RESULT_ROOT.mkdir(parents=True, exist_ok=True)
stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
OUTPUT_DIR = RESULT_ROOT / f"{PROFILE}-{stamp}"
print("Output:", OUTPUT_DIR)


In [ ]:
cmd = [
    sys.executable, str(REPO / "scripts" / "benchmark_pdelta3_frontier_layer.py"),
    "--profile", PROFILE,
    "--layer", str(LAYER),
    "--curriculum-contexts", CURRICULUM,
    "--validation-contexts", VALIDATION_CONTEXTS,
    "--test-contexts", TEST_CONTEXTS,
    "--seed", str(SEED),
    "--output-dir", str(OUTPUT_DIR),
]
print(" ".join(cmd), flush=True)
process = subprocess.Popen(cmd, cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in process.stdout:
    print(line, end="")
return_code = process.wait()
if return_code:
    raise subprocess.CalledProcessError(return_code, cmd)


In [ ]:
import json
import pandas as pd
from IPython.display import display

selection = json.loads((OUTPUT_DIR / "selection.json").read_text())
report = json.loads((OUTPUT_DIR / "pdelta3_frontier_report.json").read_text())
validation = pd.read_csv(OUTPUT_DIR / "validation_context_summary.csv")
test = pd.read_csv(OUTPUT_DIR / "test_summary.csv")
diagnostics = pd.read_csv(OUTPUT_DIR / "candidate_diagnostics.csv")
history = pd.read_csv(OUTPUT_DIR / "training_history.csv")

print("SELECTION")
print(json.dumps(selection, indent=2))
print("\nCANDIDATE DIAGNOSTICS")
display(diagnostics.sort_values("output_nmse"))
print("\nLONG-CONTEXT VALIDATION (used for selection)")
display(validation.sort_values(["context", "delta_nll"]))
print("\nHELD-OUT TEST")
display(test.sort_values(["context", "delta_nll"]))

strict = test[test["verdict"] == "strict_quality_win"]
print("\nANY STRICT TRANSFORMER QUALITY WIN:", "YES" if len(strict) else "NO")
if len(strict):
    display(strict)


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))
for name, part in test.groupby("candidate"):
    part = part.sort_values("context")
    plt.plot(part["context"], part["delta_nll"], marker="o", label=name)
plt.axhline(0.0, linewidth=1)
plt.axhline(0.02, linewidth=1, linestyle="--")
plt.xlabel("Context length")
plt.ylabel("Candidate - Transformer NLL")
plt.title("PDelta3 frontier candidates: held-out quality gap")
plt.legend(fontsize=8)
plt.grid(alpha=0.25)
plt.show()

plt.figure(figsize=(10, 5))
for name, part in test.groupby("candidate"):
    part = part.sort_values("context")
    plt.plot(part["context"], 100 * part["state_vs_transformer_fp16"], marker="o", label=name)
plt.xlabel("Context length")
plt.ylabel("Persistent state / Transformer FP16 KV (%)")
plt.title("Persistent-state scaling")
plt.legend(fontsize=8)
plt.grid(alpha=0.25)
plt.show()


In [ ]:
import shutil
from google.colab import files

archive = shutil.make_archive(str(OUTPUT_DIR), "zip", root_dir=OUTPUT_DIR)
print("Downloading complete experiment:", archive)
files.download(archive)
